# Speed Up Basemap Ordering and Download

In [1]:
import os
import json
import requests
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
from pprint import pprint
import math
import time
import ast

In [2]:
# Get Planet API Key
%load_ext dotenv
%dotenv

api_key = os.getenv('PL_BM_API_KEY')

## Define Functions

In [3]:
def makemydir(dir_path):
    try:
        os.makedirs(dir_path)
    except OSError:
        pass

In [4]:
def recu_down(url, filename): # recurrent download with ContentTooShortError
    try:
        urllib.request.urlretrieve(url,filename)
    except urllib.error.ContentTooShortError:
        print('Download failed. Trying again...')
        recu_down(url, filename)

# Import Data

In [5]:
grids_filtered = gpd.read_file('../data/new_planet_grids_for_arts_closest_year_v.3.1.0.geojson')

In [6]:
print(grids_filtered.planet_basemap_year.sort_values().unique())
grids_filtered_annual = [
    grids_filtered[grids_filtered.planet_basemap_year == year] 
    for year 
    in grids_filtered.planet_basemap_year.sort_values().unique()
]
print([len(df.index) for df in grids_filtered_annual])

n = 5000
grids_filtered_annual_chunks = [
    [annual_grids[i:i+n] for i in range(0,annual_grids.shape[0],n)]
    for annual_grids 
    in grids_filtered_annual
    ]
pprint([[len(df.index) for df in year] for year in grids_filtered_annual_chunks])

[2016. 2017.]
[314, 128]
[[314], [128]]


# Download

In [7]:
order_info_path = '../data/download_20250715/planet_basemap_orders.csv'
order_download_path = '../data/download_20250715/planet_basemap_downloads.csv'
prior_orders = pd.read_csv(order_info_path,
                           converters = {'order_info': ast.literal_eval})
prior_orders = [row['order_info'] for idx, row in prior_orders.iterrows()]
prior_orders[0]

{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/762930a0-9042-4aab-b12b-76e69fdfa297',
  'results': [{'delivery': 'success',
    'expires_at': '2025-07-16T17:55:28.755Z',
    'location': 'https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3NTI2ODg1MjgsInN1YiI6IldyN2RrdnkxQnJmUHRkejRObFNFYW5RRUFEQWo1bGJDY0ZFTllKQjVTNmVWbW4yY3VERUFSRm1WdjJIT0cvdGI1Z0ZpaHBkQTNGOHJsdUUvWkpVQVF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS83NjI5MzBhMC05MDQyLTRhYWItYjEyYi03NmU2OWZkZmEyOTcvZ2xvYmFsX3F1YXJ0ZXJseV8yMDE2cTNfbW9zYWljLzYyNy0zMjQ3X3F1YWQudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI1MDcxNSUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1M

In [9]:
for annual_grids in grids_filtered_annual_chunks[1][0:1]:
    
    basemap_name = 'global_quarterly_' + str(int(annual_grids.planet_basemap_year.iloc[0])) + 'q3_mosaic'

    requested_list = []

    # split image info into list of chunks of approximately 5 basemaps to download
    n = 5
    grids_filtered_list = [annual_grids[i:i+n] for i in range(0,annual_grids.shape[0],n)]
    
    count = 0
    
    # submit order for each item in the list
    for chunk in grids_filtered_list:
        count += 1
        
        # Get chunk info
        item_ids = list(chunk.id)

        order_name = (
            basemap_name
            + '_'
            + item_ids[0]
            + '_'
            + item_ids[len(item_ids)-1]
        )
        
        # make directory to download data to
        dir_path = (
            '../data/download_20250715/'
            + basemap_name
            + '/'
            + order_name

        )
        makemydir(dir_path)

        order_link = [order['_links']['_self'] for order in prior_orders if order['name'] == order_name][0]
        
        try:
            order = requests.get(order_link, auth=(api_key, '')).json()
        except:
            time.sleep(5)
            order = requests.get(order_link, auth=(api_key, '')).json()

        order_id = order['id']

        for file in order['_links']['results']:
            filename = dir_path + '/' + file['name'].replace(order_id + '/', '').replace('/', '_')
            url = file['location']

            print("-------------------------------------")

            if not os.path.exists(filename):
                print("Downloading ", order_name)
                # download the file
                start_time = time.time()
                try:
                    recu_down(url, filename)# the actual download
                except:
                    time.sleep(5)
                    recu_down(url, filename)# the actual download

                elapsed_time = time.time() - start_time
                print("downloading time =", np.round(elapsed_time, 2), "seconds")

                # save info about images that have been downloaded
                download_df = pd.DataFrame({
                    'order_name': [order_name],
                    'filename': filename,
                    'url': url
                })

                download_df.to_csv(
                    order_download_path,
                    index = False,
                    mode = 'a',
                    header = not os.path.exists(order_download_path)
                )
            else:
                print(order_name, ' already downloaded.')

            print("-------------------------------------")
            print("\n")

-------------------------------------
downloading time = 2.42 seconds
-------------------------------------


-------------------------------------
downloading time = 0.33 seconds
-------------------------------------


-------------------------------------
downloading time = 0.31 seconds
-------------------------------------


-------------------------------------
downloading time = 0.31 seconds
-------------------------------------


-------------------------------------
downloading time = 2.33 seconds
-------------------------------------


-------------------------------------
downloading time = 0.34 seconds
-------------------------------------


-------------------------------------
downloading time = 0.35 seconds
-------------------------------------


-------------------------------------
downloading time = 0.31 seconds
-------------------------------------


-------------------------------------
downloading time = 2.75 seconds
-------------------------------------


----------